In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

print(f"Pandas version: {pd.__version__}")
print("Day 4 - pandas")

Pandas version: 3.0.3
Day 4 - pandas


In [6]:
# Simulate metadata for 15 documents across your RAG pipeline
# This is exactly what you'd store in a database alongside ChromaDB

random.seed(42)
np.random.seed(42)

sources = ["rag_paper.pdf", "llm_guide.pdf", "vector_db.pdf", "fine_tuning.pdf"]
authors = ["Jay", "Alice", "Bob", "Carol"]
statuses = ["ingested", "embedded", "failed", "pending"]

# Generate realistic data
n_docs = 15
data = {
    "doc_id": [f"doc_{i:03}" for i in range(n_docs)],
    "source": [random.choice(sources) for _ in range(n_docs)],
    "author": [random.choice(authors) for _ in range(n_docs)],
    "page_number": [random.randint(1, 50) for _ in range(n_docs)],
    "word_count": [random.randint(50, 500) for _ in range(n_docs)],
    "status": [random.choice(statuses) for _ in range(n_docs)],
    "similarity_score": [round(random.uniform(0, 1), 4) for _ in range(n_docs)],
    "created_at":[
        datetime.now()- timedelta(days=random.randint(0,30))
        for _ in range(n_docs)
    ]
}

df = pd.DataFrame(data)

print("=== Document Metadata DataFrame ===\n")
print(df.to_string())
print(f"\nShape: {df.shape}")
print(f"\nColumn types:\n {df.dtypes}")
      



=== Document Metadata DataFrame ===

     doc_id           source author  page_number  word_count    status  similarity_score                 created_at
0   doc_000    rag_paper.pdf  Alice           25         474  embedded            0.4623 2026-05-16 09:25:44.389072
1   doc_001    rag_paper.pdf  Carol            7         371  ingested            0.2699 2026-05-02 09:25:44.389072
2   doc_002    vector_db.pdf  Alice           23         366   pending            0.9254 2026-05-07 09:25:44.389072
3   doc_003    llm_guide.pdf  Carol           23         491    failed            0.6882 2026-05-10 09:25:44.389072
4   doc_004    llm_guide.pdf    Bob           39         235   pending            0.2196 2026-04-24 09:25:44.389072
5   doc_005    llm_guide.pdf    Jay           17         345    failed            0.3243 2026-04-23 09:25:44.389072
6   doc_006    rag_paper.pdf  Alice            3         148  embedded            0.7683 2026-05-02 09:25:44.389072
7   doc_007    rag_paper.pdf  Carol

In [7]:
print("==== Basci Inspection ===\n")

# First 5 rows
print("Head:")
print(df.head())

# Last 3 rows
print("\n Tail: ")
print(df.tail(3))

# Statistical summary
print("\n Summary Statistics: ")
print(df.describe())

# Check for missing values
print("\n Missing values per column:")
print(df.isnull().sum())

# Value counts  - how many docs per status
print("\n Documents per status: ")
print(df["status"].value_counts())

# Value counts  - how many docs per source
print("\n Documents per source: ")
print(df["source"].value_counts())



==== Basci Inspection ===

Head:
    doc_id         source author  page_number  word_count    status  \
0  doc_000  rag_paper.pdf  Alice           25         474  embedded   
1  doc_001  rag_paper.pdf  Carol            7         371  ingested   
2  doc_002  vector_db.pdf  Alice           23         366   pending   
3  doc_003  llm_guide.pdf  Carol           23         491    failed   
4  doc_004  llm_guide.pdf    Bob           39         235   pending   

   similarity_score                 created_at  
0            0.4623 2026-05-16 09:25:44.389072  
1            0.2699 2026-05-02 09:25:44.389072  
2            0.9254 2026-05-07 09:25:44.389072  
3            0.6882 2026-05-10 09:25:44.389072  
4            0.2196 2026-04-24 09:25:44.389072  

 Tail: 
     doc_id         source author  page_number  word_count    status  \
12  doc_012  llm_guide.pdf    Bob            6         445  embedded   
13  doc_013  llm_guide.pdf    Jay           36         198  embedded   
14  doc_014  rag_pape

In [8]:
print("=== Filtering ===\n")

# Single condition - all embedded documents
embedded = df[df["status"]=="embedded"]
print(f"Embedded docs({len(embedded)}):")
print(embedded[["doc_id", "source", "similarity_score"]])

# Multiple conditions - high scoring embedded docs
high_quality = df[
    (df["status"] == "embedded") &
    (df["similarity_score"] > 0.5)
]
print(f"\nHigh quality embedded docs({len(high_quality)}):")
print(high_quality[["doc_id", "source", "similarity_score"]])

# OR condition - failed or pending docs (need reprocessing)
needs_work = df[
    (df["status"]=="failed")|
    (df["status"]== "pending")
]
print(f"\nDocs needing reprocessing ({len(needs_work)}):")
print(needs_work[["doc_id", "source", "status"]])

# isin - filter multiple values at once
not_ready =df[df["status"].isin(["failed", "pending", "ingested"])]
print(f"\nDocs not ready for retrieval ({len(not_ready)}):")
print(not_ready[["doc_id", "status"]])

# String filter - docs from rag_paper
rag_docs = df[df["source"].str.contains("rag")]
print(f"\n RAG paper docs ({len(rag_docs)}):")
print(rag_docs[["doc_id", "source", "word_count"]])

=== Filtering ===

Embedded docs(6):
     doc_id         source  similarity_score
0   doc_000  rag_paper.pdf            0.4623
6   doc_006  rag_paper.pdf            0.7683
9   doc_009  rag_paper.pdf            0.8050
12  doc_012  llm_guide.pdf            0.9131
13  doc_013  llm_guide.pdf            0.5672
14  doc_014  rag_paper.pdf            0.7179

High quality embedded docs(5):
     doc_id         source  similarity_score
6   doc_006  rag_paper.pdf            0.7683
9   doc_009  rag_paper.pdf            0.8050
12  doc_012  llm_guide.pdf            0.9131
13  doc_013  llm_guide.pdf            0.5672
14  doc_014  rag_paper.pdf            0.7179

Docs needing reprocessing (7):
     doc_id           source   status
2   doc_002    vector_db.pdf  pending
3   doc_003    llm_guide.pdf   failed
4   doc_004    llm_guide.pdf  pending
5   doc_005    llm_guide.pdf   failed
7   doc_007    rag_paper.pdf   failed
8   doc_008  fine_tuning.pdf   failed
10  doc_010    rag_paper.pdf   failed

Docs not 

In [11]:
print("=== GroupBy Analysis ===\n")

# Average similarity score per status
print("Average similarity score by status:")
print(df.groupby("status")["similarity_score"]. mean().round(4))

# Word count stats per source
print("\n Word count stats by source:")
print(df.groupby("source")["word_count"].agg(["mean", "min", "max", "count"]).round(1))

# Count documents per author per status
print("\nDocuments per author per status: ")
print(df.groupby(["author", "status" ]).size().unstack(fill_value=0))

# Total word count per source
print("\nTotal words ingested per source:")
total_words=df.groupby("source")["word_count"].sum().sort_values(ascending=False)
print(total_words)

# Which author has highest average simiarity score?
print("\nAverage similarity score per author:")
author_scores = df.groupby("author")["similarity_score"].mean().sort_values(ascending=False)
print(author_scores.round(4))
print(f"\nBest performing author: {author_scores.index[0]}")

=== GroupBy Analysis ===

Average similarity score by status:
status
embedded    0.7056
failed      0.4583
ingested    0.1680
pending     0.5725
Name: similarity_score, dtype: float64

 Word count stats by source:
                  mean  min  max  count
source                                 
fine_tuning.pdf   85.0   85   85      1
llm_guide.pdf    342.8  198  491      5
rag_paper.pdf    265.0   73  474      8
vector_db.pdf    366.0  366  366      1

Documents per author per status: 
status  embedded  failed  ingested  pending
author                                     
Alice          2       1         1        1
Bob            2       1         0        1
Carol          0       2         1        0
Jay            2       1         0        0

Total words ingested per source:
source
rag_paper.pdf      2120
llm_guide.pdf      1714
vector_db.pdf       366
fine_tuning.pdf      85
Name: word_count, dtype: int64

Average similarity score per author:
author
Bob      0.6899
Jay      0.5365
Al

In [12]:
print("=== Apply and Transform ===")

# Add a quality label based on similarity score
def quality_label(score: float) -> str:
    if score >= 0.7:
        return "high"
    elif score >= 0.4:
        return "medium"
    else:
        return "low"

df["quality"] = df["similarity_score"].apply(quality_label)
print("Quality distribution: ")
print(df["quality"].value_counts())

# Add a text length category
df["size_category"] = df["word_count"].apply(
    lambda x: "large" if x > 300 else "medium" if x > 150 else "small"
)
print("\n Size category distribution:")
print(df["size_category"].value_counts())

# Add days since creation
df["days_old"] = (datetime.now() - df["created_at"]).dt.days
print("\n Days old stats: ")
print(df["days_old"].describe().round(1))

# Add a ready_for_retrieval flag
df["ready"] = (df["status"] == "embedded") & (df["quality"] == "high")
print(f"\nDocs ready for retrieval : {df['ready'].sum()}")
print(df[df["ready"]][["doc_id", "source", "similarity_score", "quality"]])

=== Apply and Transform ===
Quality distribution: 
quality
high      6
low       5
medium    4
Name: count, dtype: int64

 Size category distribution:
size_category
large     8
small     4
medium    3
Name: count, dtype: int64

 Days old stats: 
count    15.0
mean     14.9
std       8.2
min       4.0
25%       7.5
50%      15.0
75%      20.0
max      29.0
Name: days_old, dtype: float64

Docs ready for retrieval : 4
     doc_id         source  similarity_score quality
6   doc_006  rag_paper.pdf            0.7683    high
9   doc_009  rag_paper.pdf            0.8050    high
12  doc_012  llm_guide.pdf            0.9131    high
14  doc_014  rag_paper.pdf            0.7179    high


In [16]:
class DocumentMetadataManager:
    """ 
    Manages document metadata using pandas.
    In your real project this wraps a SQLite or PostgreSQL table.
    """

    def __init__(self):
        self.df = pd.DataFrame(columns =[
            "doc_id", "source", "author", "word_count",
            "status", "similarity_score", "created_at", "quality"
        ])

    def add_document(
            self, 
            doc_id:str,
            source:str,
            author:str,
            word_count:int
    )-> None:
        new_row = {
            "doc_id": doc_id,
            "source": source,
            "author": author,
            "word_count": word_count,
            "status": "pending",
            "similarity_score": None,
            "created_at": datetime.now(),
            "quality": None
        }
        self.df = pd.concat(
            [self.df, pd.DataFrame([new_row])],
            ignore_index=True
        )
        print(f"[ADDED] {doc_id} from {source}")

    def update_status(self, doc_id:str, status:str)-> None:
        self.df.loc[self.df["doc_id"] == doc_id, "status"] = status
        print(f"[UPDATED {doc_id} -> {status}")

    def update_score(self, doc_id: str, score: float) -> None:
        self.df.loc[self.df["doc_id"] == doc_id, "similarity_score"] = score
        quality = "high" if score >= 0.7 else "medium" if score >= 0.4 else "low"
        self.df.loc[self.df["doc_id"] == doc_id, "quality"] = quality

    def get_pending(self) -> pd.DataFrame:
        return self.df[self.df["status"]=="pending"]

    def get_ready(self) -> pd.DataFrame:
        return self.df[
            (self.df["status"]=="embedded")&
            (self.df["quality"]== "high")
            ]

    def pipeline_health(self) -> dict:
        total = len(self.df)
        if total ==0:
            return {"error": "No Documenta"}
        return{
            "total":total,
            "by_status": self.df["status"].value_counts().to_dict(),
            "avg_score": round(
                self.df["similarity_score"].dropna().mean(), 4
            ),
            "ready_for_retrieval": len(self.get_ready())
        }
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __str__(self)-> str:
        return f"DocumentMetadataManager({len(self)} documents)"
    

# Test it
from typing import Dict

print("=== Document Metadata Manager\n")
manager = DocumentMetadataManager()

# Add documents
manager.add_document("doc_001", "rag_paper.pdf", "Jay", 450)
manager.add_document("doc_002", "llm_guide.pdf", "Jay", 320)
manager.add_document("doc_003", "vector_db.pdf", "Jay", 180)
manager.add_document("doc_004", "rag_paper.pdf", "Jay", 95)

#Simulate pipeline processing
print("\n=== Processing Pipeline ===")
manager.update_status("doc_001", "ingested")
manager.update_status("doc_001", "embedded")
manager.update_score("doc_001", 0.89)

manager.update_status("doc_002", "ingested")
manager.update_status("doc_002", "embedded")
manager.update_score("doc_002", 0.45)

manager.update_status("doc_003", "failed")
# doc_004 stays pending

# Check pipeline health
print("\n=== Pipeline Health ===")
health = manager.pipeline_health()
for key, value in health.items():
    print(f"   {key}: {value}")

# Get pending docs
print(f"\n=== Pending docs===")
pending = manager.get_pending()
print(pending[["doc_id", "source", "status"]])

# Get ready docs
print(f"\n=== Ready for Retrieval ===")
ready = manager.get_ready()
print(ready[["doc_id", "source", "similarity_score", "quality" ]])

print(f"\n{manager}")



=== Document Metadata Manager

[ADDED] doc_001 from rag_paper.pdf
[ADDED] doc_002 from llm_guide.pdf
[ADDED] doc_003 from vector_db.pdf
[ADDED] doc_004 from rag_paper.pdf

=== Processing Pipeline ===
[UPDATED doc_001 -> ingested
[UPDATED doc_001 -> embedded
[UPDATED doc_002 -> ingested
[UPDATED doc_002 -> embedded
[UPDATED doc_003 -> failed

=== Pipeline Health ===
   total: 4
   by_status: {'embedded': 2, 'failed': 1, 'pending': 1}
   avg_score: 0.67
   ready_for_retrieval: 1

=== Pending docs===
    doc_id         source   status
3  doc_004  rag_paper.pdf  pending

=== Ready for Retrieval ===
    doc_id         source similarity_score quality
0  doc_001  rag_paper.pdf             0.89    high

DocumentMetadataManager(4 documents)
